# CWSB — version_rbn

**v1 is never touched.** This notebook only reads it, to get a "before" number.

### The problem, already measured on v1

| test | v1 |
|---|---|
| infested photo, **middle blacked out** (no stem), still called infested | **78%** |
| infested photo, **blurred to 64×64** (no hole can exist), still called infested | **42%** |
| infested photo, full | 92% |
| healthy photo, false alarm | 4% |

v1 does not need the stem. It recognises the **scene**.

### Why

```
photo 3456 x 4608  ->  model sees 384 x 384   =  9x shrink
a 100 px bore hole ->  11 px                  =  invisible
```

The network's finest view is a 12×12 grid. On a whole photo one grid cell covers
288 px of your original, so a 100 px hole is a **third of one cell** — it can
never be isolated.

### The fix being tested

Cut each photo into overlapping pieces (half the short edge), score every piece,
and take the **highest**. Shrink drops to 4.5×, the hole becomes 22 px, and one
grid cell now covers 144 px — the hole **fills a cell**.

The winning piece is also the **location**, with no labels and no YOLO.

Same ConvNeXt-Tiny. Same weights. Same recipe. **Only the input changes.**


## 1 · Settings

**Before running anything: Runtime → Change runtime type → T4 GPU.**
Changing it later wipes every file.

In [ ]:
FOLD   = 0        # 0, 1 or 2. Run all three and average for the honest number.
TOPK   = 4        # round 2 keeps this many best pieces per infested photo
EPOCHS = 15
PATIENCE = 5
USE_DRIVE = True  # False = upload the two files from your computer

!nvidia-smi -L || echo 'NO GPU - Runtime > Change runtime type > T4 GPU, then rerun'
%pip -q install timm
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn the GPU on first.'

## 2 · Upload

Put **two files** at the **top level of MyDrive** (not inside a folder):

| File | Size | Why |
|---|---|---|
| `cwsb_package.zip` | 190 MB | the photos (768 px short side) **and** the plant split |
| `contender_B.pt` | 111 MB | v1 — used only to print the "before" number |

You already have both. `cwsb_package.zip` is the file you used to train v1.
`contender_B.pt` is in your `models/v1/` folder.

**Why not `old_data.zip` (2.9 GB)?** The package photos are already 768 px on the
short side, so a piece is 384 px and needs no shrink at all. Cutting a 1728 px
piece from the full photo and shrinking it to 384 gives **exactly the same
detail**. Same result, 15× smaller upload.

In [ ]:
import shutil, zipfile
from pathlib import Path

BASE = Path('/content/cwsb')
shutil.rmtree(BASE, ignore_errors=True)
(BASE / 'version_rbn/src').mkdir(parents=True)
(BASE / 'models/v1').mkdir(parents=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    mydrive = Path('/content/drive/MyDrive')
    pkg = mydrive / 'cwsb_package.zip'
    assert pkg.exists(), 'cwsb_package.zip not found at the top level of MyDrive'
    shutil.copy(pkg, '/content/cwsb_package.zip')
    old = mydrive / 'contender_B.pt'
    if old.exists():
        shutil.copy(old, BASE / 'models/v1/contender_B.pt')
else:
    from google.colab import files
    print('Select cwsb_package.zip AND contender_B.pt (you can pick both):')
    for name in files.upload():
        dest = ('/content/cwsb_package.zip' if name.endswith('.zip')
                else BASE / 'models/v1' / name)
        shutil.move('/content/' + name, dest)

with zipfile.ZipFile('/content/cwsb_package.zip') as zf:
    zf.extractall(BASE)

n = len(list((BASE / 'work/clean').glob('*.jpg')))
has_split = (BASE / 'work/splits/split.csv').exists()
has_v1 = (BASE / 'models/v1/contender_B.pt').exists()
print(f'photos in work/clean : {n}')
print(f'split.csv            : {"yes" if has_split else "MISSING"}')
print(f'v1 for the baseline  : {"yes" if has_v1 else "not uploaded (baseline will be skipped)"}')
assert n > 0 and has_split, 'the package is not what this notebook expects'

from PIL import Image
p = next((BASE / 'work/clean').glob('*.jpg'))
w, h = Image.open(p).size
print(f'example photo size   : {w} x {h}   ->  piece side will be {int(min(w,h)*0.5)} px')

## 3 · The scripts

Six files. Each one is a step you can read and run on its own.

In [ ]:
%%writefile /content/cwsb/version_rbn/src/rbn_common.py
"""Shared helpers for version_rbn.

Everything here is READ-ONLY with respect to v1. Nothing in version_rbn ever
writes into models/v1/, src/, colab/, old_data/ or work/.

The one idea of version_rbn: do not feed the whole photo. Feed overlapping
pieces, and score a photo as the HIGHEST piece score.
"""
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from torchvision import transforms

# --- the geometry of a piece -------------------------------------------------
PIECE_FRAC = 0.50     # piece side = half the SHORT edge of the photo
OVERLAP = 0.50        # neighbouring pieces share half their area
IMG = 384             # what the network eats

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
BACKBONE = "convnext_tiny.fb_in22k_ft_in1k_384"

# A piece is already square, so we only resize it to 384. No centre crop -
# a centre crop would throw away the edge of the piece, and the damage may be
# exactly there.
#
# LANCZOS on purpose: cut_pieces.py saves the training pieces with LANCZOS, so
# scoring a live photo must shrink them the same way. If training used LANCZOS
# and serving used bilinear, the model would see slightly different pictures
# than it was taught on - a small train/serve mismatch that is easy to avoid.
RESAMPLE = transforms.InterpolationMode.LANCZOS

eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG), interpolation=RESAMPLE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# v1's own evaluation transform, copied from src/train.py where the comment says
# it "MUST stay identical in the API". A whole frame is NOT square, so squashing
# it to 384x384 would show v1 a picture it was never trained or served on, and
# the baseline numbers would not be v1's real behaviour. Used only for
# --mode whole, which exists to measure v1.
whole_tf = transforms.Compose([
    transforms.Resize(IMG + 32),
    transforms.CenterCrop(IMG),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.65, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.30, 0.30, 0.30, 0.05),   # kills the camera's colour
    transforms.RandomGrayscale(p=0.10),               # kills colour as a shortcut
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
])


def piece_boxes(w: int, h: int, frac: float = PIECE_FRAC, overlap: float = OVERLAP):
    """Top-left corners of overlapping square pieces covering the whole photo."""
    side = max(int(min(w, h) * frac), 32)
    stride = max(int(side * (1.0 - overlap)), 1)

    def starts(total):
        xs = list(range(0, max(total - side, 0) + 1, stride))
        if total > side and xs[-1] != total - side:
            xs.append(total - side)       # always cover the far edge
        return xs

    return side, [(x, y) for y in starts(h) for x in starts(w)]


def pieces_of(pil):
    """Cut a PIL photo into its pieces. Returns (side, boxes, list of PIL pieces)."""
    w, h = pil.size
    side, boxes = piece_boxes(w, h)
    return side, boxes, [pil.crop((x, y, x + side, y + side)) for x, y in boxes]


def load_model(ckpt_path, device=None):
    """Load any checkpoint written by this project. Never writes to the file."""
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ck = torch.load(Path(ckpt_path), map_location="cpu", weights_only=False)
    model = timm.create_model(ck.get("backbone", BACKBONE), pretrained=False, num_classes=2)
    model.load_state_dict(ck["state_dict"])
    model.eval().to(dev)
    return model, ck, dev


@torch.no_grad()
def score_whole(model, pil, dev):
    """v1 style: one score for the whole frame, using v1's own eval transform."""
    x = whole_tf(pil).unsqueeze(0).to(dev)
    return float(torch.softmax(model(x), 1)[0, 1])


@torch.no_grad()
def score_pieces(model, pil, dev, batch=24):
    """version_rbn style. Returns (best_score, all_scores, side, boxes)."""
    side, boxes, pcs = pieces_of(pil)
    out = []
    for i in range(0, len(pcs), batch):
        x = torch.stack([eval_tf(p) for p in pcs[i:i + batch]]).to(dev)
        out.append(torch.softmax(model(x), 1)[:, 1].cpu())
    s = torch.cat(out).numpy()
    return float(s.max()), s, side, boxes


def score_photo(model, pil, dev, mode):
    """mode is 'whole' (v1) or 'pieces' (version_rbn)."""
    if mode == "whole":
        return score_whole(model, pil, dev)
    return score_pieces(model, pil, dev)[0]


def blur_down(pil, px):
    """Throw away ALL fine detail: shrink to px x px, blow back up to 384.

    At 64x64 a bore hole cannot exist. Whatever score survives this is coming
    from overall colour and layout - that is background, not disease.
    """
    return pil.resize((px, px), Image.BILINEAR).resize((IMG, IMG), Image.BILINEAR)


def blackout_centre(pil):
    """Remove the middle 50% - where the stem normally is. Background only."""
    w, h = pil.size
    cw, chh = w // 2, h // 2
    x0, y0 = (w - cw) // 2, (h - chh) // 2
    out = pil.copy()
    out.paste((0, 0, 0), (x0, y0, x0 + cw, y0 + chh))
    return out


def centre_only(pil):
    """Keep only the middle 50%."""
    w, h = pil.size
    cw, chh = w // 2, h // 2
    x0, y0 = (w - cw) // 2, (h - chh) // 2
    return pil.crop((x0, y0, x0 + cw, y0 + chh))


def balanced_accuracy(truth, score, thr=0.5):
    pred = (np.asarray(score) >= thr).astype(int)
    truth = np.asarray(truth)
    per = [(pred[truth == c] == c).mean() for c in (0, 1) if (truth == c).any()]
    return float(np.mean(per))


In [ ]:
%%writefile /content/cwsb/version_rbn/src/measure.py
"""STEP 1 and STEP 7 - does the model read the stem, or the background?

Two tests, both on photos the model was never trained to handle specially.

TEST A - blackout
    full    the photo as it is
    centre  only the middle 50% (where the stem usually is)
    border  the middle 50% BLACKED OUT - only background left
    A model that reads the stem scores 'border' low.

TEST B - detail
    the photo shrunk to 64x64 (or 32, 16) and blown back up.
    At 64x64 a bore hole cannot exist. Any score left is colour and layout.

READ-ONLY. Opens the checkpoint, prints numbers, writes nothing.

    python version_rbn/src/measure.py --ckpt models/v1/contender_B.pt
    python version_rbn/src/measure.py --ckpt models/version_rbn/rbn_fold0.pt --mode pieces
"""
import argparse
import csv
import random
import statistics
import sys
from pathlib import Path

from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parent))
import rbn_common as C

ROOT = Path(__file__).resolve().parents[2]

# the numbers v1 produced, so you always have something to compare against
V1_BASELINE = {"border": 0.78, "blur64": 0.42, "full": 0.92, "false_alarm": 0.04}
TARGET = {"border": 0.25, "blur64": 0.15, "full": 0.85, "false_alarm": 0.10}


def gather(source, n_unhealthy, seed):
    """Photo paths per class. 'clean' works in Colab, 'old_data' on your PC."""
    if source == "clean":
        with open(ROOT / "work/splits/split.csv", newline="") as f:
            rows = list(csv.DictReader(f))
        pick = {c: [ROOT / "work/clean" / r["file"] for r in rows if r["label"] == c]
                for c in ("Unhealthy", "Healthy")}
    else:
        pick = {"Unhealthy": sorted((ROOT / "old_data/Unhealthy(Infected)").glob("*.jpg")),
                "Healthy": sorted((ROOT / "old_data/Healthy").glob("*.jpg"))}
    rng = random.Random(seed)
    return {"Unhealthy": rng.sample(pick["Unhealthy"], min(n_unhealthy, len(pick["Unhealthy"]))),
            "Healthy": pick["Healthy"]}


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--mode", choices=["whole", "pieces"], default="whole",
                    help="whole = v1 style, pieces = version_rbn style (max over pieces)")
    ap.add_argument("--source", choices=["clean", "old_data"], default="clean")
    ap.add_argument("--n-unhealthy", type=int, default=40)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    model, ck, dev = C.load_model(args.ckpt)
    print(f"checkpoint : {args.ckpt}")
    print(f"backbone   : {ck.get('backbone')}   device {dev}")
    print(f"scoring    : {'MAX over pieces (version_rbn)' if args.mode == 'pieces' else 'whole frame (v1)'}")
    print(f"photos     : from {args.source}\n")

    sets = gather(args.source, args.n_unhealthy, args.seed)

    def sc(pil):
        return C.score_photo(model, pil, dev, args.mode)

    # ---------------- TEST A ----------------
    print("TEST A - blackout " + "-" * 44)
    print(f"{'class':<10}{'variant':<9}{'median':>8}{'mean':>8}{'>=0.90':>9}{'n':>5}")
    res = {}
    for cls, paths in sets.items():
        acc = {"full": [], "centre": [], "border": []}
        for p in paths:
            im = Image.open(p).convert("RGB")
            acc["full"].append(sc(im))
            acc["centre"].append(sc(C.centre_only(im)))
            acc["border"].append(sc(C.blackout_centre(im)))
        res[cls] = acc
        for k in ("full", "centre", "border"):
            v = acc[k]
            print(f"{cls:<10}{k:<9}{statistics.median(v):>8.3f}{statistics.mean(v):>8.3f}"
                  f"{sum(x >= 0.90 for x in v) / len(v):>8.0%}{len(v):>5}")

    # ---------------- TEST B ----------------
    print("\nTEST B - detail destroyed " + "-" * 36)
    print(f"{'detail kept':<16}{'infested >=0.90':>17}{'healthy >=0.90':>16}{'gap':>8}")
    blur = {}
    for px, name in ((None, "full detail"), (64, "64x64"), (32, "32x32"), (16, "16x16")):
        su = [sc(Image.open(p).convert("RGB") if px is None
                 else C.blur_down(Image.open(p).convert("RGB"), px)) for p in sets["Unhealthy"]]
        sh = [sc(Image.open(p).convert("RGB") if px is None
                 else C.blur_down(Image.open(p).convert("RGB"), px)) for p in sets["Healthy"]]
        rate = sum(x >= 0.90 for x in su) / len(su)
        blur[name] = rate
        print(f"{name:<16}{rate:>16.0%}{sum(x >= 0.90 for x in sh) / len(sh):>16.0%}"
              f"{statistics.median(su) - statistics.median(sh):>8.2f}")

    # ---------------- verdict ----------------
    def rate(cls, var):
        v = res[cls][var]
        return sum(x >= 0.90 for x in v) / len(v)

    got = {"border": rate("Unhealthy", "border"), "full": rate("Unhealthy", "full"),
           "false_alarm": rate("Healthy", "full"), "blur64": blur["64x64"]}

    print("\nSCORECARD " + "-" * 51)
    print(f"{'test':<34}{'v1':>7}{'now':>8}{'target':>9}{'':>6}")
    lines = [("infested, background only", "border", "<"),
             ("infested, 64x64 blur", "blur64", "<"),
             ("infested, full photo", "full", ">"),
             ("healthy, false alarm", "false_alarm", "<")]
    allpass = True
    for label, key, direction in lines:
        ok = got[key] < TARGET[key] if direction == "<" else got[key] > TARGET[key]
        allpass &= ok
        print(f"{label:<34}{V1_BASELINE[key]:>6.0%}{got[key]:>8.0%}"
              f"{direction + f'{TARGET[key]:.0%}':>9}{'  PASS' if ok else '  FAIL':>6}")
    print("-" * 61)
    print("VERDICT:", "ALL FOUR PASS - the model stopped using the background"
          if allpass else "NOT YET - see the FAIL lines above")
    print("\nNote: all four must pass together. A model that answers 'healthy' to")
    print("everything passes the first two lines and is useless - lines 3 and 4")
    print("exist to catch exactly that.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/cwsb/version_rbn/src/make_folds.py
"""STEP 2 - honest test groups.

You have 9 healthy plants, and v1's validation set had only 2 healthy PHOTOS.
A result decided by 2 photos is luck, not measurement.

So the plants are rotated through 3 folds. In fold k:

    TEST  = block k of plants            (never seen during training)
    VAL   = a quarter of what is left    (used to stop training)
    TRAIN = the rest

Every plant is tested exactly once across folds 0, 1, 2.

THE RULE THAT NEVER BREAKS: all photos of one plant stay in the same set.
If photo 1 of a plant is in train and photo 2 is in test, the model has already
seen that plant and the test score is a lie. The script aborts if that happens.

Reads work/splits/split.csv. Writes only version_rbn/work/folds.csv.
"""
import argparse
import csv
import random
from collections import defaultdict
from pathlib import Path

HERE = Path(__file__).resolve().parents[1]
ROOT = HERE.parent
SPLIT = ROOT / "work" / "splits" / "split.csv"
OUT = HERE / "work" / "folds.csv"
N_FOLDS = 3
VAL_SHARE = 0.25
MIN_VAL_PHOTOS = 3     # early stopping on 1 or 2 photos is meaningless
SEED = 42


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--folds", type=int, default=N_FOLDS)
    args = ap.parse_args()

    with open(SPLIT, newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["split"] != "excluded"]

    plants = defaultdict(list)                      # group -> rows
    label_of = {}
    for r in rows:
        plants[r["group"]].append(r)
        label_of[r["group"]] = r["label"]

    by_class = defaultdict(list)
    for g in plants:
        by_class[label_of[g]].append(g)

    print(f"photos: {len(rows)}")
    for c, gs in by_class.items():
        print(f"  {c:10s} {len(gs):3d} plants, {sum(len(plants[g]) for g in gs):3d} photos")

    # Plants are very uneven: one healthy plant holds 11 of 28 photos, one
    # infested plant holds 120 of 468. Dealing plants one by one would leave a
    # fold with almost no healthy photos to train on. So we balance by PHOTO
    # COUNT: biggest plant first, always into the fold that is currently
    # emptiest.
    rng = random.Random(SEED)
    size = {g: len(rs) for g, rs in plants.items()}
    blocks = {c: [[] for _ in range(args.folds)] for c in by_class}
    for c, gs in by_class.items():
        gs = sorted(gs, key=lambda g: (-size[g], g))
        tot = [0] * args.folds
        for g in gs:
            i = min(range(args.folds), key=lambda j: tot[j])
            blocks[c][i].append(g)
            tot[i] += size[g]
        print(f"  {c:10s} test blocks hold {tot} photos")

    out_rows = []
    for k in range(args.folds):
        test_g = {g for c in blocks for g in blocks[c][k]}

        # Validation: take the SMALLEST remaining plants until ~20% of the
        # remaining photos are covered. Smallest first, so validation never
        # swallows the one big plant that training needs.
        val_g = set()
        for c in by_class:
            rest = sorted([g for g in by_class[c] if g not in test_g],
                          key=lambda g: (size[g], g))
            want = max(MIN_VAL_PHOTOS, round(sum(size[g] for g in rest) * VAL_SHARE))
            budget = max(1, len(rest) - 2)          # always leave >=2 plants to train on
            chosen, took = [], 0
            for g in rest[:budget]:                 # smallest plants first
                if took >= want:
                    break
                chosen.append(g)
                took += size[g]
            if took < MIN_VAL_PHOTOS:
                # Too few photos this way (one big plant holds nearly all of
                # them). Take the smallest single plant that is big enough -
                # early stopping on 1 photo is worthless.
                big = [g for g in rest if size[g] >= MIN_VAL_PHOTOS]
                if big and budget >= 1:
                    chosen, took = [big[0]], size[big[0]]   # smallest plant big enough
            val_g.update(chosen)

        for g, grows in plants.items():
            s = "test" if g in test_g else ("val" if g in val_g else "train")
            for r in grows:
                out_rows.append({"file": r["file"], "label": r["label"], "group": g,
                                 "fold": k, "split": s})

        # --- leakage check, per fold ---
        where = defaultdict(set)
        for r in out_rows:
            if r["fold"] == k:
                where[r["group"]].add(r["split"])
        bad = {g: s for g, s in where.items() if len(s) > 1}
        if bad:
            print(f"LEAKAGE in fold {k}: {bad}")
            return 1

        cnt = defaultdict(lambda: defaultdict(int))
        pl = defaultdict(lambda: defaultdict(set))
        for r in out_rows:
            if r["fold"] == k:
                cnt[r["split"]][r["label"]] += 1
                pl[r["split"]][r["label"]].add(r["group"])
        print(f"\nfold {k}")
        for s in ("train", "val", "test"):
            h, u = cnt[s]["Healthy"], cnt[s]["Unhealthy"]
            ph, pu = len(pl[s]["Healthy"]), len(pl[s]["Unhealthy"])
            print(f"  {s:6s} healthy {h:3d} photos / {ph} plants   "
                  f"unhealthy {u:3d} photos / {pu} plants")

    OUT.parent.mkdir(parents=True, exist_ok=True)
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "label", "group", "fold", "split"])
        w.writeheader()
        w.writerows(out_rows)
    print(f"\nLEAKAGE CHECK OK - no plant is in two sets in any fold")
    print(f"-> {OUT}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/cwsb/version_rbn/src/cut_pieces.py
"""STEP 4 - cut every photo into overlapping pieces.

Why this is the whole point:

    whole photo 3456 px -> 384 px  = 9.0x shrink -> a 100 px bore hole = 11 px
    one piece   1728 px -> 384 px  = 4.5x shrink -> the same hole      = 22 px

The network's finest view is a 12 x 12 grid. On a whole photo one grid cell
covers 288 px of the original, so a 100 px hole is a THIRD of one cell - it can
never be isolated. On a piece, one cell covers 144 px, so the hole fills a cell.

Note: if the photos came from cwsb_package.zip they are already 768 px on the
short side, so a piece is 384 px and needs no shrink at all. The detail that
reaches the network is identical either way.

Pieces are cut ONCE and reused by every fold.
Reads work/clean. Writes only version_rbn/work/pieces/.
"""
import argparse
import csv
import sys
from collections import Counter
from pathlib import Path

from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parent))
import rbn_common as C

HERE = Path(__file__).resolve().parents[1]
ROOT = HERE.parent
CLEAN = ROOT / "work" / "clean"
FOLDS = HERE / "work" / "folds.csv"
OUT = HERE / "work" / "pieces"


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--quality", type=int, default=92)
    args = ap.parse_args()

    if not FOLDS.exists():
        print(f"missing {FOLDS} - run make_folds.py first")
        return 1

    with open(FOLDS, newline="") as f:
        rows = list(csv.DictReader(f))
    photos = {}
    for r in rows:
        photos.setdefault(r["file"], {"label": r["label"], "group": r["group"]})
    todo = sorted(photos)
    if args.limit:
        todo = todo[: args.limit]

    OUT.mkdir(parents=True, exist_ok=True)
    man, per, sizes = [], [], Counter()

    for i, fname in enumerate(todo, 1):
        src = CLEAN / fname
        if not src.exists():
            continue
        img = Image.open(src).convert("RGB")
        side, boxes, pcs = C.pieces_of(img)
        sizes[side] += 1
        stem = Path(fname).stem
        meta = photos[fname]
        dest = OUT / meta["label"]
        dest.mkdir(parents=True, exist_ok=True)

        for k, (p, (x, y)) in enumerate(zip(pcs, boxes)):
            if p.size != (C.IMG, C.IMG):
                p = p.resize((C.IMG, C.IMG), Image.LANCZOS)
            name = f"{stem}__p{k:02d}.jpg"
            p.save(dest / name, "JPEG", quality=args.quality)
            man.append({"file": f"{meta['label']}/{name}", "photo": fname,
                        "label": meta["label"], "group": meta["group"],
                        "piece": k, "x": x, "y": y, "side": side})
        per.append(len(boxes))
        if i % 50 == 0:
            print(f"  {i}/{len(todo)} photos -> {len(man)} pieces")

    with open(OUT.parent / "pieces.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "photo", "label", "group",
                                          "piece", "x", "y", "side"])
        w.writeheader()
        w.writerows(man)

    byc = Counter(m["label"] for m in man)
    print(f"\n{len(per)} photos -> {len(man)} pieces "
          f"(avg {sum(per) / max(len(per), 1):.1f} per photo, "
          f"min {min(per)}, max {max(per)})")
    for k, v in byc.items():
        print(f"  {k:10s} {v:6d} pieces")
    print(f"  piece sides seen: {dict(sizes.most_common(4))}")
    print(f"-> {OUT.parent / 'pieces.csv'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/cwsb/version_rbn/src/train_rbn.py
"""STEP 5, 6, 8 - train version_rbn in two rounds, then set the thresholds.

ROUND 1  every piece carries its photo's label. This is wrong on purpose:
         most pieces of a sick plant show clean bark. Round 1 only has to be
         good enough to guess WHICH pieces hold the damage.

ROUND 2  score every training piece with the round-1 model.
           sick photo    -> keep only its TOPK best pieces
           healthy photo -> keep ALL pieces (every part of a healthy plant is
                            healthy, so every piece is a true sample)
         then throw round 1 away and train again from ImageNet weights.

This is Multiple Instance Learning: "the bag is positive, but I do not know
which item inside it is".

Early stopping watches PHOTO accuracy scored as max-over-pieces - the way the
model will actually be used. Never piece accuracy.

Thresholds come from HEALTHY validation scores. Never from infected scores:
that exact mistake already happened in this project and collapsed both lines
to 0.91.

Writes only models/version_rbn/ .
"""
import argparse
import csv
import json
import random
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

sys.path.insert(0, str(Path(__file__).resolve().parent))
import rbn_common as C

HERE = Path(__file__).resolve().parents[1]
ROOT = HERE.parent
PIECES = HERE / "work" / "pieces"
MODELS = ROOT / "models" / "version_rbn"
SEED = 42


class PieceSet(Dataset):
    def __init__(self, rows, tf):
        self.rows, self.tf = rows, tf

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        im = Image.open(PIECES / r["file"]).convert("RGB")
        return self.tf(im), 1 if r["label"] == "Unhealthy" else 0


def load_rows(fold):
    with open(HERE / "work" / "pieces.csv", newline="") as f:
        pieces = list(csv.DictReader(f))
    with open(HERE / "work" / "folds.csv", newline="") as f:
        split_of = {r["file"]: r["split"] for r in csv.DictReader(f)
                    if int(r["fold"]) == fold}
    out = defaultdict(list)
    for p in pieces:
        s = split_of.get(p["photo"])
        if s:
            out[s].append(p)
    return out


@torch.no_grad()
def piece_scores(model, rows, dev, bs=48):
    model.eval()
    dl = DataLoader(PieceSet(rows, C.eval_tf), batch_size=bs, num_workers=2, pin_memory=True)
    out = []
    for x, _ in dl:
        with torch.autocast("cuda", enabled=(dev == "cuda")):
            out.append(torch.softmax(model(x.to(dev, non_blocking=True)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def to_photo(rows, scores):
    """max over a photo's pieces -> (photo names, scores, truths)"""
    best, truth = defaultdict(float), {}
    for r, s in zip(rows, scores):
        best[r["photo"]] = max(best[r["photo"]], float(s))
        truth[r["photo"]] = 1 if r["label"] == "Unhealthy" else 0
    names = sorted(best)
    return names, np.array([best[n] for n in names]), np.array([truth[n] for n in names])


def fit(train_rows, val_rows, dev, epochs, patience, tag):
    torch.manual_seed(SEED)
    n_h = sum(r["label"] == "Healthy" for r in train_rows)
    n_u = len(train_rows) - n_h
    print(f"  train pieces: {n_h} healthy + {n_u} unhealthy")

    # balance the batches (17:1 imbalance) AND weight the loss - v1's recipe
    wts = [1.0 / max(n_h, 1) if r["label"] == "Healthy" else 1.0 / max(n_u, 1)
           for r in train_rows]
    sampler = WeightedRandomSampler(wts, num_samples=len(train_rows), replacement=True)
    dl = DataLoader(PieceSet(train_rows, C.train_tf), batch_size=16, sampler=sampler,
                    num_workers=2, pin_memory=True, drop_last=True)

    cls_w = torch.tensor([len(train_rows) / (2 * max(n_h, 1)),
                          len(train_rows) / (2 * max(n_u, 1))],
                         dtype=torch.float32, device=dev)
    model = timm.create_model(C.BACKBONE, pretrained=True, num_classes=2).to(dev)
    crit = nn.CrossEntropyLoss(weight=cls_w)
    scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))

    best, best_state, bad, hist = -1.0, None, 0, []
    for ep in range(1, epochs + 1):
        warm = ep <= 2
        for p in model.parameters():
            p.requires_grad = not warm
        for p in model.get_classifier().parameters():
            p.requires_grad = True
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=1e-3 if warm else 1e-4, weight_decay=0.05)
        model.train()
        tot = 0.0
        for x, y in dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=(dev == "cuda")):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item() * x.size(0)

        _, ps, py = to_photo(val_rows, piece_scores(model, val_rows, dev))
        acc = C.balanced_accuracy(py, ps)
        hist.append({"epoch": ep, "loss": tot / len(dl.dataset), "val_photo_bal_acc": acc})
        print(f"  epoch {ep:2d} {'(warm-up)' if warm else '         '} "
              f"loss {tot / len(dl.dataset):.4f}   val photo bal-acc {acc:.3f}")
        if acc > best + 1e-4:
            best, bad = acc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"  early stop at epoch {ep}  (best {best:.3f})")
                break

    model.load_state_dict(best_state)
    MODELS.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": best_state, "backbone": C.BACKBONE, "img": C.IMG,
                "scoring": "max over pieces", "piece_frac": C.PIECE_FRAC,
                "overlap": C.OVERLAP, "val_photo_bal_acc": best, "history": hist},
               MODELS / f"{tag}.pt")
    print(f"  -> {MODELS / (tag + '.pt')}   best val photo bal-acc {best:.3f}")
    return model, best


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--fold", type=int, default=0)
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--patience", type=int, default=5)
    ap.add_argument("--topk", type=int, default=4)
    ap.add_argument("--skip-round2", action="store_true")
    args = ap.parse_args()

    random.seed(SEED)
    np.random.seed(SEED)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if dev == "cpu":
        print("WARNING: no GPU. This will take many hours - use Colab with a T4.")

    R = load_rows(args.fold)
    print(f"fold {args.fold}: pieces  train {len(R['train'])}  "
          f"val {len(R['val'])}  test {len(R['test'])}")

    print(f"\n=== ROUND 1 - every piece keeps its photo's label ===")
    model, _ = fit(R["train"], R["val"], dev, args.epochs, args.patience,
                   f"rbn_fold{args.fold}_round1")

    if not args.skip_round2:
        print(f"\n=== ROUND 2 - keep the top {args.topk} pieces of each sick photo ===")
        s = piece_scores(model, R["train"], dev)
        by = defaultdict(list)
        for i, r in enumerate(R["train"]):
            by[r["photo"]].append((float(s[i]), i))
        keep = []
        for photo, lst in by.items():
            if R["train"][lst[0][1]]["label"] == "Healthy":
                keep += [i for _, i in lst]
            else:
                keep += [i for _, i in sorted(lst, reverse=True)[: args.topk]]
        tr2 = [R["train"][i] for i in sorted(keep)]
        print(f"  {len(R['train'])} pieces -> {len(tr2)} pieces")
        model, _ = fit(tr2, R["val"], dev, args.epochs, args.patience,
                       f"rbn_fold{args.fold}")
    else:
        import shutil
        shutil.copy(MODELS / f"rbn_fold{args.fold}_round1.pt", MODELS / f"rbn_fold{args.fold}.pt")

    # ---------- STEP 8: thresholds from HEALTHY validation scores ----------
    _, vs, vy = to_photo(R["val"], piece_scores(model, R["val"], dev))
    healthy = np.sort(vs[vy == 0])
    if len(healthy):
        t_low = float(min(0.60, max(0.35, healthy.max() + 0.05)))
    else:
        t_low = 0.50
    t_high = 0.90
    print(f"\n=== THRESHOLDS (from {len(healthy)} healthy validation photos) ===")
    print(f"  healthy val scores: {np.round(healthy, 3).tolist()}")
    print(f"  T_low  = {t_low:.2f}   (just above the healthy band)")
    print(f"  T_high = {t_high:.2f}")

    # ---------- held-out TEST, photo level ----------
    names, ts, ty = to_photo(R["test"], piece_scores(model, R["test"], dev))
    print(f"\n=== HELD-OUT TEST, photo level (score = max over pieces) ===")
    for thr in (t_low, t_high):
        pred = (ts >= thr).astype(int)
        tp = int(((pred == 1) & (ty == 1)).sum()); fn = int(((pred == 0) & (ty == 1)).sum())
        fp = int(((pred == 1) & (ty == 0)).sum()); tn = int(((pred == 0) & (ty == 0)).sum())
        print(f"  at {thr:.2f}: caught {tp:3d}  missed {fn:3d}  "
              f"false alarms {fp:3d}  cleared {tn:3d}  bal-acc {C.balanced_accuracy(ty, ts, thr):.3f}")

    cfg = {"model": f"rbn_fold{args.fold}.pt", "backbone": C.BACKBONE, "img": C.IMG,
           "scoring": "max over pieces", "piece_frac": C.PIECE_FRAC, "overlap": C.OVERLAP,
           "T_low": round(t_low, 3), "T_high": t_high,
           "plant_score": "max over the plant's photos",
           "calibrated_on": "healthy validation photos of this fold only",
           "fold": args.fold, "topk": args.topk,
           "note": "version_rbn scores a photo as the HIGHEST of its pieces. "
                   "Serving MUST cut pieces the same way. v1 is untouched."}
    (MODELS / f"config_fold{args.fold}.json").write_text(json.dumps(cfg, indent=1),
                                                         encoding="utf-8")
    (MODELS / f"test_scores_fold{args.fold}.json").write_text(json.dumps(
        [{"photo": n, "score": float(x), "truth": int(t)} for n, x, t in zip(names, ts, ty)],
        indent=1), encoding="utf-8")
    print(f"\n-> {MODELS / f'config_fold{args.fold}.json'}")
    print(f"NEXT: measure.py --ckpt models/version_rbn/rbn_fold{args.fold}.pt --mode pieces")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/cwsb/version_rbn/src/look.py
"""STEP 9 - look with your own eyes.

One PDF page per photo:

   +------------------------+   +--------------------+
   | whole photo            |   | that piece, zoomed |
   |      +------+          |   |                    |
   |      |yellow| <- the   |   |   big and clear    |
   |      +------+   piece  |   |                    |
   +------------------------+   +--------------------+

If the yellow box sits on bark damage, the model is real.
If it sits on soil or leaves, the numbers lied to you.

Wrong answers are placed FIRST, so you cannot skip them.

Writes only version_rbn/reports/.

    python version_rbn/src/look.py --fold 0
    python version_rbn/src/look.py --folder some/photos
"""
import argparse
import csv
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image, ImageOps

sys.path.insert(0, str(Path(__file__).resolve().parent))
import rbn_common as C

HERE = Path(__file__).resolve().parents[1]
ROOT = HERE.parent
MODELS = ROOT / "models" / "version_rbn"
REPORTS = HERE / "reports"
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def verdict(s, lo, hi):
    return "wsb_likely" if s >= hi else ("no_visible_signs" if s < lo else "inconclusive")


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--fold", type=int, default=0)
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--folder", default=None, help="score any folder instead of the test split")
    ap.add_argument("--max-pages", type=int, default=60)
    args = ap.parse_args()

    ckpt = Path(args.ckpt) if args.ckpt else MODELS / f"rbn_fold{args.fold}.pt"
    cfg_path = MODELS / f"config_fold{args.fold}.json"
    cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {"T_low": 0.5, "T_high": 0.9}
    lo, hi = float(cfg["T_low"]), float(cfg["T_high"])

    model, ck, dev = C.load_model(ckpt)
    print(f"model      : {ckpt}")
    print(f"scoring    : MAX over pieces   thresholds {lo:.2f} / {hi:.2f}")

    # ---- which photos ----
    if args.folder:
        base = Path(args.folder)
        items = [(p, None) for p in sorted(base.rglob("*")) if p.suffix.lower() in EXTS]
        title = base.name
    else:
        with open(HERE / "work" / "folds.csv", newline="") as f:
            rows = [r for r in csv.DictReader(f)
                    if int(r["fold"]) == args.fold and r["split"] == "test"]
        items = [(ROOT / "work/clean" / r["file"], r["label"]) for r in rows]
        title = f"fold{args.fold}_test"
    print(f"photos     : {len(items)}\n")

    recs = []
    for p, truth in items:
        im = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
        best, scores, side, boxes = C.score_pieces(model, im, dev)
        k = int(scores.argmax())
        v = verdict(best, lo, hi)
        mark = ("-" if truth is None else
                "inconclusive" if v == "inconclusive" else
                "CORRECT" if (v == "wsb_likely") == (truth == "Unhealthy") else "WRONG")
        recs.append({"file": p.name, "truth": truth, "score": best, "verdict": v,
                     "mark": mark, "n": len(boxes), "box": boxes[k], "side": side, "im": im})
        print(f"  {p.name:<46s} {best:.3f}  {v:<17s} {mark}")

    REPORTS.mkdir(parents=True, exist_ok=True)
    csv_path = REPORTS / f"{title}_rbn.csv"
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file", "truth", "score", "verdict", "mark", "pieces", "box_x", "box_y", "side"])
        for r in recs:
            w.writerow([r["file"], r["truth"] or "", f"{r['score']:.4f}", r["verdict"],
                        r["mark"], r["n"], r["box"][0], r["box"][1], r["side"]])

    order = ([r for r in recs if r["mark"] == "WRONG"] +
             [r for r in recs if r["mark"] == "inconclusive"] +
             [r for r in recs if r["mark"] not in ("WRONG", "inconclusive")])
    pdf_path = REPORTS / f"{title}_rbn.pdf"
    with PdfPages(pdf_path) as pdf:
        dec = [r for r in recs if r["mark"] in ("CORRECT", "WRONG")]
        L = [f"version_rbn - fold {args.fold}", "",
             f"scoring     : highest of {recs[0]['n'] if recs else 0} pieces per photo",
             f"thresholds  : < {lo:.2f} no signs | inconclusive | >= {hi:.2f} wsb likely",
             f"photos      : {len(recs)}", ""]
        if dec:
            L += [f"correct     : {sum(r['mark'] == 'CORRECT' for r in dec)} of {len(dec)} decided",
                  f"wrong       : {sum(r['mark'] == 'WRONG' for r in dec)}",
                  f"inconclusive: {sum(r['mark'] == 'inconclusive' for r in recs)}", ""]
        L += ["Wrong answers are the first pages. Look at the yellow box:",
              "on bark damage = real. On soil or leaves = the number lied."]
        fig = plt.figure(figsize=(10, 5.4))
        fig.text(0.05, 0.95, "\n".join(L), va="top", family="monospace", fontsize=9)
        plt.axis("off"); pdf.savefig(fig, dpi=90); plt.close(fig)

        for r in order[: args.max_pages]:
            im = r["im"]
            th = im.copy(); th.thumbnail((900, 900))
            sc = th.width / im.width
            bx, by = r["box"]
            fig, ax = plt.subplots(1, 2, figsize=(11, 5.2))
            ax[0].imshow(th)
            ax[0].add_patch(mpatches.Rectangle((bx * sc, by * sc), r["side"] * sc,
                                               r["side"] * sc, fill=False,
                                               edgecolor="yellow", linewidth=2.5))
            ax[0].set_title("photo + the piece that decided", fontsize=10)
            crop = im.crop((bx, by, bx + r["side"], by + r["side"]))
            crop.thumbnail((700, 700))
            ax[1].imshow(crop)
            ax[1].set_title("that piece, zoomed in", fontsize=10)
            for a in ax:
                a.axis("off")
            col = {"CORRECT": "#2e7d32", "WRONG": "#b3261e",
                   "inconclusive": "#a05a00", "-": "#333333"}[r["mark"]]
            t = f'{r["file"]}   score {r["score"]:.3f}  ->  {r["verdict"]}'
            if r["truth"]:
                t += f'   true: {r["truth"]}   [{r["mark"]}]'
            fig.suptitle(t, color=col, fontsize=11)
            pdf.savefig(fig, dpi=85); plt.close(fig)

    print(f"\n-> {csv_path}")
    print(f"-> {pdf_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 4 · STEP 1 — measure v1 (the "before" number)

**Read only.** It opens v1, shows it photos, prints numbers. It writes nothing.

Two tests:

```
   FULL                CENTRE              BORDER
+---------+         +---------+         +---------+
|  ,,,,,  |         |         |         |  ,,,,,  |
| , STEM ,|         |  STEM   |         | ,#####, |
|  ,,,,,  |         |         |         |  ,#####,|
+---------+         +---------+         +---------+
 everything        only the stem      STEM REMOVED
```

plus the same photos blurred down to 64×64, where a bore hole cannot exist.

Takes about 5 minutes.

In [ ]:
import os, subprocess
from pathlib import Path
os.chdir('/content/cwsb')
if Path('models/v1/contender_B.pt').exists():
    subprocess.run(['python', 'version_rbn/src/measure.py',
                    '--ckpt', 'models/v1/contender_B.pt',
                    '--mode', 'whole', '--source', 'clean'], check=True)
else:
    print('contender_B.pt not uploaded - skipping the baseline.')
    print('From your PC it was: background-only 78%, 64x64 blur 42%, full 92%, false alarm 4%.')

## 5 · STEP 2 — honest test groups

You have **9 healthy plants**, and v1's validation set had only **2 healthy
photos**. A result decided by 2 photos is luck, not measurement.

Two of your healthy plants hold 19 of the 28 photos, and one infested plant
holds 120 of 468 — so plants are dealt out **biggest first, always into the
emptiest fold**, balancing by photo count.

**The rule that never breaks:** every photo of one plant stays in one set.
The script aborts if a plant ever lands in two.

In [ ]:
import os
os.chdir('/content/cwsb')
!python version_rbn/src/make_folds.py

## 6 · STEP 4 — cut the photos into pieces

About **18–21 pieces per photo**, overlapping by half so a hole is never cut in
two. Pieces are cut once and reused by all three folds. Takes a few minutes.

In [ ]:
import os
os.chdir('/content/cwsb')
!python version_rbn/src/cut_pieces.py

## 7 · STEPS 5 & 6 — train, two rounds

**Round 1** gives every piece its photo's label. That is wrong on purpose —
most pieces of a sick plant show clean bark. Round 1 only has to be good enough
to guess *which* pieces hold the damage.

**Round 2** keeps the top 4 pieces of each infested photo, keeps **all** pieces
of healthy photos (every part of a healthy plant is healthy), and trains again
from ImageNet weights.

Early stopping watches **photo** accuracy scored as max-over-pieces — the way
the model will really be used, never piece accuracy.

**About 1 to 2 hours on a T4.** Do not close the tab.

In [ ]:
import os
os.chdir('/content/cwsb')
!python version_rbn/src/train_rbn.py --fold {FOLD} --epochs {EPOCHS} --patience {PATIENCE} --topk {TOPK}

## 8 · STEP 7 — measure again. This is the verdict.

The **same two tests** from Step 1, now on version_rbn.

| test | v1 | must become |
|---|---|---|
| infested, background only | 78% | **under 25%** |
| infested, 64×64 blur | 42% | **under 15%** |
| infested, full photo | 92% | above 85% |
| healthy, false alarm | 4% | under 10% |

**All four together.** A lazy model that answers "healthy" to everything passes
the first two lines and is useless — lines 3 and 4 exist to catch exactly that.

In [ ]:
import os
os.chdir('/content/cwsb')
!python version_rbn/src/measure.py --ckpt models/version_rbn/rbn_fold{FOLD}.pt --mode pieces --source clean

## 9 · STEP 9 — look with your own eyes

One page per held-out test photo. Wrong answers first, so you cannot skip them.

```
+------------------------+   +--------------------+
| whole photo            |   | that piece, zoomed |
|      +------+          |   |                    |
|      |yellow|          |   |   big and clear    |
|      +------+          |   |                    |
+------------------------+   +--------------------+
```

Yellow box on bark damage → the model is real.
Yellow box on soil or leaves → the numbers lied to you.

In [ ]:
import os
os.chdir('/content/cwsb')
!python version_rbn/src/look.py --fold {FOLD} --max-pages 60

## 10 · Download everything

In [ ]:
import os, shutil
os.chdir('/content/cwsb')
shutil.make_archive('/content/version_rbn_fold%d' % FOLD, 'zip', '.', 'models/version_rbn')
shutil.make_archive('/content/version_rbn_reports', 'zip', 'version_rbn', 'reports')
from google.colab import files
print('1) the model + config + test scores')
files.download('/content/version_rbn_fold%d.zip' % FOLD)
print('2) the PDF and CSV')
files.download('/content/version_rbn_reports.zip')

## What the result means

**If all four lines PASS** — version_rbn decides from a small region and can show
where. Compare its caught / missed / false alarms with v1 before replacing
anything. Then run FOLD = 1 and FOLD = 2 and average the three.

**If it FAILS** — the problem is not the code, it is the photos:

- you have **9 healthy plants**
- **268 infested photos** come from the `FIELD` camera, and **zero healthy
  photos** come from that camera

So "this is a FIELD photo" already means "infested". No training trick removes
a clue that strong.

The fix is one sentence:

> **Photograph healthy plants the same way the infested ones were photographed.**
> Same camera, same distance, same angle, same farms, close to the stem.
> 100–200 plants.

That single action is worth more than everything in this notebook. If your team
lead asks what you need, that is the answer.

---

**v1 was never opened for writing.** Check it on your PC afterwards:

```
config.json          badeae29570e98f6a127455acbf06eec
contender_B.pt       f63f0c25a012006f84fcef9efd9edf59
cwsb_v1.pt           f63f0c25a012006f84fcef9efd9edf59
cwsb_v1.onnx         452863794c733678b5c4ea742a993cc4
```

Same fingerprints = untouched. That is maths, not a promise.
